In [1]:
import os
import numpy as np
import rasterio
import cv2
import pandas as pd
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

def read_png_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(path)
    return (mask > 0).astype(np.uint8)

def read_tif_mask(path):
    with rasterio.open(path) as src:
        mask = src.read(1)
    return (mask > 0).astype(np.uint8)

def load_prediction(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".png":
        return read_png_mask(path)
    elif ext in [".tif", ".tiff"]:
        return read_tif_mask(path)
    else:
        raise ValueError(f"Unsupported format: {path}")

def load_gt_mask(name):
    mask_path_1 = f'../GEE_Masks/GEE_resized/test_gee/NDWI_Mask_{name}_resized.tif'
    mask_path_2 = f'../GEE_Masks/GEE_resized/train_gee/train_gee/NDWI_Mask_{name}_resized_corrupt.tif'

    if os.path.exists(mask_path_1):
        path = mask_path_1
    elif os.path.exists(mask_path_2):
        path = mask_path_2
    else:
        raise FileNotFoundError(f"No GT mask for {name}")

    with rasterio.open(path) as src:
        mask = src.read(1)

    return (mask > 0).astype(np.uint8)

def ensemble_consensus(pred_masks):
    """
    pred_masks: list of binary masks (H, W)
    """
    stack = np.stack(pred_masks, axis=0)  # (K, H, W)
    votes = np.sum(stack, axis=0)
    return (votes >= (len(pred_masks) / 2)).astype(np.uint8)

def compute_ssim(a, b):
    """
    a, b : binary masks (0/1), same shape
    """
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    # Optional: multiply by 255 for standard SSIM range
    return ssim(a, b, data_range=1.0)

def compute_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    if union == 0:
        return np.nan
    return inter / union

def consensus_iou_for_image(name, pred_paths, visualize=True):
    preds = []

    for p in pred_paths:
        preds.append(load_prediction(p))

    consensus = ensemble_consensus(preds)
    gt = load_gt_mask(name)
    
    iou_score = compute_iou(consensus, gt)
    
    if visualize:
        num_preds = len(preds)
        fig, axes = plt.subplots(1, num_preds + 2, figsize=(3*(num_preds+2), 3))
        
        # Show GT
        axes[0].imshow(gt, cmap='gray')
        axes[0].set_title('GT Mask')
        axes[0].axis('off')
        
        # Show predictions
        for i, pred in enumerate(preds):
            axes[i+1].imshow(pred, cmap='gray')
            axes[i+1].set_title(f'Model {i+1}')
            axes[i+1].axis('off')
        
        # Show consensus
        axes[-1].imshow(consensus, cmap='gray')
        axes[-1].set_title('Consensus')
        axes[-1].axis('off')
        
        plt.suptitle(f'{name} | IoU vs GT: {iou_score:.3f}')
        plt.tight_layout()
        plt.savefig(f"./outputs_iou/{name}.png")
        plt.close() 

    return iou_score

def consensus_ssim_for_image(name, pred_paths, visualize=False):
    """
    Compute SSIM between ensemble consensus and GT mask.
    Optionally visualize GT, predictions, and consensus.
    
    name: image name
    pred_paths: list of paths to 4 model predictions
    visualize: if True, show matplotlib plot
    """
    
    preds = [load_prediction(p) for p in pred_paths]
    
    consensus = ensemble_consensus(preds)
    gt = load_gt_mask(name)
    
    ssim_score = compute_ssim(consensus, gt)
    
    if visualize:
        num_preds = len(preds)
        fig, axes = plt.subplots(1, num_preds + 2, figsize=(3*(num_preds+2), 3))
        
        # Show GT
        axes[0].imshow(gt, cmap='gray')
        axes[0].set_title('GT Mask')
        axes[0].axis('off')
        
        # Show predictions
        for i, pred in enumerate(preds):
            axes[i+1].imshow(pred, cmap='gray')
            axes[i+1].set_title(f'Model {i+1}')
            axes[i+1].axis('off')
        
        # Show consensus
        axes[-1].imshow(consensus, cmap='gray')
        axes[-1].set_title('Consensus')
        axes[-1].axis('off')
        
        plt.suptitle(f'{name} | SSIM vs GT: {ssim_score:.3f}')
        plt.tight_layout()
        plt.savefig(f"./outputs/{name}.png")
        plt.close() 
        # plt.show()
    
    print(f"{name} done")
    return ssim_score

In [2]:
df = pd.read_csv('Coordinates_GEE(in).csv')
consensus_ssims = []
consensus_ious = []

for _, row in df.iterrows():
    name = row['name']

    # YOU define this list per image
    pred_paths = [
        f'../GEE_Output/UNet/0_all/dense_{name}.tif',
        f'../GEE_Output/DeepLabOutputs/0_all/Outputs/pred_{name}.png',
        f'../GEE_Output/Segformer/0_all/Outputs/pred_{name}.png',
        f'../GEE_Output/Maskformer/0_all/Outputs/pred_{name}.png'
    ]

    try:
        ssim_val = consensus_ssim_for_image(name, pred_paths, visualize=True)
        iou = consensus_iou_for_image(name, pred_paths)
        consensus_ssims.append(ssim_val)
        consensus_ious.append(iou)
        
    except Exception as e:
        print(f"{name}: {e}")
        consensus_ssims.append(np.nan)
        consensus_ious.append(np.nan)

df['consensus_ssim'] = consensus_ssims
df['consensus_iou'] = consensus_ious

# Rank most suspicious GT masks
df_sorted = df.sort_values('consensus_ssim', ascending=False)
df_sorted1 = df.sort_values('consensus_iou', ascending=False)
df_sorted.to_csv("Coordinates_Consensus_SSIM.csv", index=False)
df_sorted1.to_csv("Coordinates_Consensus_IoU.csv", index=False)

c:\Users\ADMIN\anaconda3\envs\tf_gpu\lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


0 done
1 done
2 done
3 done
4 done
5 done
6 done
7 done
8 done
9 done
10 done
11 done
12 done
13 done
14 done
15 done
16 done
17 done
18 done
19 done
20 done
21 done
22 done
23 done
24 done
25 done
26 done
27 done
28 done
29 done
30 done
31 done
32 done
33 done
34 done
35 done
36 done
37 done
38 done
39 done
40 done
41 done
42 done
43 done
44 done
45 done
46 done
47 done
48 done
49 done
50 done
51 done
52 done
53 done
54 done
55 done
56 done
57 done
58 done
59 done
60 done
61 done
62 done
63 done
64 done
65 done
66 done
67 done
68 done
69 done
70 done
71 done
72 done
73 done
74 done
75 done
76 done
77 done
78 done
79 done
80 done
81 done
82 done
83 done
84 done
85 done
86 done
87 done
88 done
89 done
90 done
91 done
92 done
93 done
94 done
95 done
96 done
97 done
98 done
99 done
100 done
101 done
102 done
103 done
104 done
105 done
106 done
107 done
108 done
109 done
110 done
111 done
112 done
113 done
114 done
115 done
116 done
117 done
118 done
119 done
120 done
121 done
122 done
123

In [ ]:
# df_ssim = pd.read_csv("Coordinates_Consensus_SSIM.csv")

# bins = [0.0, 0.4, 0.6, 0.75, 0.9, 1.0]
# labels = ["Very Low", "Low", "Medium", "High", "Very High"]

# df_ssim["ssim_group"] = pd.cut(
#     df_ssim["consensus_ssim"],
#     bins=bins,
#     labels=labels,
#     include_lowest=True
# )

# df_ssim["ssim_group"].value_counts().sort_index()



In [4]:
# df_ssim.sort_values('name', inplace=True)
# df_ssim.loc[df_ssim["consensus_ssim"] < 0.75, "correct"] = "No"
# df_ssim.to_csv("Coordinates_GEE_corrected.csv", columns=["left","right","top","bottom","name","correct","consensus_ssim"], index=False)

# df_ssim["correct"].value_counts()

In [5]:
# df["correct"].value_counts()